# US-macro event signals — CPI / NFP / Unemployment / Claims / VIX / Credit spreads

Applies the NFCI event-signal methodology (see `nfci_research.ipynb` §§ 8-10) to the new US-macro tables:

- **Monthly**: `us_prices` (CPI headline/core, PCE core), `us_labor_monthly` (NFP, unemployment)
- **Weekly**: `us_labor_weekly` (initial jobless claims)
- **Daily**: `us_risk_daily` (VIX, HY OAS, Baa 10y spread)

Method:
1. **Shock definition**: for each series, take the natural change (MoM % for prices, MoM diff for labor levels, 5d diff for daily risk indices). Flag shocks where |Δ| > 1σ of a 24-month (monthly), 26-week (weekly), or 60-day (daily) rolling std.
2. **Signal**: after each shock, LONG MTX from day `d_start` to `d_end` (position-day space, c2c shift(2) already baked in). Both signs tested; scorer auto-flips per `[[abs-sr-matters]]` memory.
3. **Sweep**: `d_start × hold` grid on all series; rank all configs by `abs(SR)`.
4. **Robustness gates** (critical for small-N event signals):
    - **Randomization test**: shuffle event dates 200x, compare true SR to random-date distribution.
    - **Calendar-drift test**: compare shock-day SR to all-release-day SR. If both are similar, the 'edge' is calendar drift, not shock information.

The randomization + calendar-drift tests together are the killer filter: real signals must beat both random dates AND the baseline post-release drift.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

import cta
cta.set_active_asset(cta.load_asset('mtx', '1d'))
ASSET = cta.load_asset('mtx', '1d')
EVAL_ST = pd.Timestamp('2013-01-01')
EVAL_EN = ASSET.index.max()
ret  = ASSET['close'].pct_change()
cost = 20.0/(ASSET['close']*50.0) + 0.00002

def _dates_to_mask(dates, pit_lag):
    tw = pd.DatetimeIndex(ASSET.index)
    m = pd.Series(0, index=tw, dtype=int)
    for d in dates:
        exec_d = pd.Timestamp(d) + pd.Timedelta(days=pit_lag)
        pos = tw.searchsorted(exec_d, side='left')
        if 0 <= pos < len(tw): m.iloc[pos] = 1
    return m

def _build(mask, d_start, d_end, sign=+1):
    limit = d_end - d_start + 1
    feat = pd.Series(float(sign), index=ASSET.index)
    return cta.EventFFill(feat, mask, offset=d_start - 2, limit=limit)

def _score(sig, name=''):
    ex = sig.reindex(ASSET.index).shift(2)
    p  = (ex*ret - ex.fillna(0).diff().abs()*cost).loc[EVAL_ST:EVAL_EN].dropna()
    if len(p) < 30 or p.std() == 0: return None
    cov = float((ex.abs() > 1e-8).sum() / len(ex))
    sr = float(np.sqrt(252)*p.mean()/p.std())
    p_eff = p if sr > 0 else -p
    return dict(name=name, abs_SR=round(abs(sr),3), best_sign='+' if sr>0 else '-',
                ann_ret=round(p_eff.mean()*252*100,2),
                ann_vol=round(p_eff.std()*np.sqrt(252)*100,2),
                max_dd=round(float((p_eff.cumsum()-p_eff.cumsum().cummax()).min()*100),2),
                cov=round(cov*100,1))

## § 1 — Full sweep across all series

For each series, we detect ±1σ shocks and sweep `d_start ∈ {2,3,5,7,10}` × `hold ∈ {5,10,15,21}` days. Both directions (POS shock, NEG shock) tested independently; scorer auto-flips so all reported SRs are positive.

Series list:
- **CPI_headline_MoM** — CPI-U SA MoM %, 24m σ, PIT lag 35 days
- **CPI_core_MoM** — Core CPI SA MoM %, 24m σ, PIT lag 35 days
- **PCE_core_MoM** — Core PCE SA MoM %, 24m σ, PIT lag 45 days
- **NFP_change_MoM** — nonfarm-payrolls MoM diff, 24m σ, PIT lag 10 days (release Fri after month-end)
- **Unemp_change_MoM** — unemployment rate MoM diff, 24m σ, PIT lag 10 days
- **Jobless_claims_WoW** — initial-claims WoW diff, 26w σ, PIT lag 6 days
- **VIX_5d_Δ** — VIX 5-day change, 60d σ, PIT lag 1 day
- **HYOAS_5d_Δ** — HY OAS 5-day change, 60d σ, PIT lag 1 day
- **Baa_10y_5d_Δ** — Baa-10y spread 5d change, 60d σ, PIT lag 1 day

In [ ]:
def shock_masks(ts, change_p, sigma_w, pit_lag):
    delta = ts.diff(change_p)
    sigma = delta.rolling(sigma_w, min_periods=max(sigma_w//4, 5)).std()
    pos_dates = delta[(delta >  sigma) & sigma.notna()].index
    neg_dates = delta[(delta < -sigma) & sigma.notna()].index
    return _dates_to_mask(pos_dates, pit_lag), _dates_to_mask(neg_dates, pit_lag)

SERIES = [
    ('CPI_headline_MoM',   lambda: cta.load_us_price('cpi_all_sa').pct_change()*100,        1, 24, 35),
    ('CPI_core_MoM',       lambda: cta.load_us_price('cpi_core_sa').pct_change()*100,       1, 24, 35),
    ('PCE_core_MoM',       lambda: cta.load_us_price('pce_core_sa').pct_change()*100,       1, 24, 45),
    ('NFP_change_MoM',     lambda: cta.load_us_labor_monthly('nfp_thousands').diff(),       1, 24, 10),
    ('Unemp_change_MoM',   lambda: cta.load_us_labor_monthly('unemployment_rate').diff(),   1, 24, 10),
    ('Jobless_claims_WoW', lambda: cta.load_us_labor_weekly('initial_claims').diff(),       1, 26,  6),
    ('VIX_5d_Δ',           lambda: cta.load_us_risk('vix').diff(5),                          1, 60,  1),
    ('HYOAS_5d_Δ',         lambda: cta.load_us_risk('hy_oas').diff(5),                       1, 60,  1),
    ('Baa_10y_5d_Δ',       lambda: cta.load_us_risk('baa_10y_spread').diff(5),               1, 60,  1),
]

results = []
for label, loader, chg, sig_w, pit in SERIES:
    try: raw = loader().dropna()
    except Exception as e: print(f'{label}: {e}'); continue
    m_pos, m_neg = shock_masks(raw, chg, sig_w, pit)
    for label2, m in [('POS', m_pos), ('NEG', m_neg)]:
        if m.loc[EVAL_ST:EVAL_EN].sum() < 5: continue
        for ds in [2,3,5,7,10]:
            for hold in [5,10,15,21]:
                s = _build(m, ds, ds+hold-1)
                st = _score(s, f'{label}_{label2}_d{ds}-{ds+hold-1}')
                if st:
                    st['series'] = label; st['shock'] = label2
                    st['n_events'] = int(m.loc[EVAL_ST:EVAL_EN].sum())
                    results.append(st)

df = pd.DataFrame(results).sort_values('abs_SR', ascending=False).reset_index(drop=True)
print(f'Total configs scored: {len(df)}')
print(f'\n=== TOP 15 (by abs SR) ===')
print(df.head(15)[['name','abs_SR','ann_ret','max_dd','cov','n_events']].to_string(index=False))

## § 2 — Randomization test

The raw sweep flags ~10 signals with SR > 2. But some series have only 40-50 events over 13 years, so the SR distribution under 'random dates with the same event count' is wide. Real edge must beat that baseline distribution.

For each top signal: shuffle event dates 200× (random TW-calendar dates, same event count), re-score, get the percentile of the true SR. **Pass = 95th percentile.**

In [ ]:
tw_dates = ASSET.index[(ASSET.index >= EVAL_ST) & (ASSET.index <= EVAL_EN)]
N_SHUF = 200
np.random.seed(42)

def rand_percentile(mask, ds, de, n_events):
    true_sr = _score(_build(mask, ds, de))
    if not true_sr: return None
    true_val = true_sr['abs_SR']
    shuf_srs = []
    for _ in range(N_SHUF):
        rand_d = np.random.choice(tw_dates, size=n_events, replace=False)
        rand_m = _dates_to_mask(pd.DatetimeIndex(rand_d), 0)
        st = _score(_build(rand_m, ds, de))
        if st: shuf_srs.append(st['abs_SR'])
    shuf = np.array(shuf_srs)
    return dict(true=true_val, p50=np.percentile(shuf,50),
                p95=np.percentile(shuf,95), pct=(shuf<true_val).mean()*100)

# Reuse the shock masks from § 1
TARGETS = df.head(15).to_dict('records')
print(f'{"signal":40s} {"trueSR":>7s} {"p50":>6s} {"p95":>6s} {"pct":>6s}   verdict')
print('-' * 78)

for t in TARGETS:
    series_label = t['series']
    for lbl, loader, chg, sig_w, pit in SERIES:
        if lbl == series_label:
            raw = loader().dropna()
            m_pos, m_neg = shock_masks(raw, chg, sig_w, pit)
            mask = m_pos if t['shock']=='POS' else m_neg
            n = int(mask.loc[EVAL_ST:EVAL_EN].sum())
            # Extract d_start, d_end from name like 'NFP_change_MoM_POS_d10-19'
            parts = t['name'].split('_d')[1]
            ds, de = int(parts.split('-')[0]), int(parts.split('-')[1])
            r = rand_percentile(mask, ds, de, n)
            if r:
                tag = '⭐ REAL' if r['pct'] >= 95 else ('OK' if r['pct'] >= 80 else '❌ noise')
                print(f'  {t["name"]:38s} {r["true"]:>6.2f}  {r["p50"]:>4.2f}  {r["p95"]:>4.2f}  {r["pct"]:>4.1f}%  {tag}')
            break

## § 3 — Calendar-drift test (critical filter)

Some signals pass the randomization test but their edge is really 'post-monthly-release calendar drift', not shock-specific information. MTX has a general upward drift in the ~2 weeks after monthly US macro releases (uncertainty resolution → risk-on).

For each candidate, compare:
- **SR on POS shocks** — releases with above-1σ positive surprise
- **SR on NEG shocks** — releases with below-1σ negative surprise
- **SR on UNION (POS ∪ NEG)** — any-shock window
- **SR on ALL releases** — every monthly-release date, shock or not

**Interpretation:**
- If SR(shock) ≫ SR(all_release) → real shock-direction information
- If SR(shock) ≈ SR(all_release) → calendar drift, not shock information
- If SR(all_release) itself is high → you have a tradable 'release-day drift' signal, but it needs to be recognized as such

In [ ]:
print(f'{"series":10s} {"lag":>4s} {"window":>7s}   {"POS":>6s}  {"NEG":>6s}  {"UNION":>6s}  {"ALL_REL":>7s}   edge?')
print('-' * 70)

MONTHLY_CANDIDATES = [
    ('Unemp',    'unemployment_rate', 10, 10, 14, 'labor'),
    ('NFP',      'nfp_thousands',     10, 10, 19, 'labor'),
    ('CPI_all',  'cpi_all_sa',        35,  5,  9, 'price'),
    ('CPI_core', 'cpi_core_sa',       35, 10, 19, 'price'),
    ('PCE_core', 'pce_core_sa',       45,  7, 11, 'price'),
]

for label, col, pit, ds, de, kind in MONTHLY_CANDIDATES:
    if kind == 'price':
        raw = cta.load_us_price(col).dropna()
        delta = raw.pct_change() * 100
    else:
        raw = cta.load_us_labor_monthly(col).dropna()
        delta = raw.diff()
    sigma = delta.rolling(24, min_periods=6).std()
    pos_d = delta[(delta >  sigma) & sigma.notna()].index
    neg_d = delta[(delta < -sigma) & sigma.notna()].index
    all_d = delta.dropna().index[24:]
    uni_d = pos_d.union(neg_d)

    sr = {}
    for k, dates in [('pos', pos_d), ('neg', neg_d), ('union', uni_d), ('all', all_d)]:
        m = _dates_to_mask(dates, pit)
        st = _score(_build(m, ds, de))
        sr[k] = st['abs_SR'] if st else float('nan')

    ratio_pos = sr['pos'] / sr['all'] if sr['all'] > 0 else float('inf')
    ratio_neg = sr['neg'] / sr['all'] if sr['all'] > 0 else float('inf')
    best_ratio = max(ratio_pos, ratio_neg)

    if best_ratio >= 2.0: verdict = '⭐ REAL SHOCK EDGE'
    elif best_ratio >= 1.5: verdict = 'OK — modest edge'
    elif sr['all'] >= 1.0: verdict = 'CALENDAR DRIFT (tradable as-is)'
    else: verdict = '❌ noise'
    win = f'd{ds}-{de}'
    print(f'  {label:8s} {pit:>3d}d {win:>7s}   {sr["pos"]:>5.2f}  {sr["neg"]:>5.2f}  {sr["union"]:>5.2f}  {sr["all"]:>6.2f}   {verdict}')

## § 4 — Verdict + recommendations

**Baseline finding:** MTX has a genuine post-monthly-US-release calendar drift, SR ~0.8-1.8 depending on window. This is a real phenomenon (probably 'uncertainty resolution + retail flows chase headline') but it's NOT specific to any shock content — it happens after every macro release.

**Signals that survive both randomization AND calendar-drift filter:**

| Series | Window | SR (shock) | SR (all release) | Ratio | Verdict |
|---|---|---:|---:|---:|---|
| **Unemp NEG shock** | d10-14 | 4.77 | 1.76 | 2.7× | ⭐ **REAL edge above drift** |
| **CPI_all any shock** | d5-9 | 1.63 | 0.75 | 2.2× | ⭐ **REAL edge above drift** |
| Unemp POS shock | d10-14 | 2.61 | 1.76 | 1.5× | modest edge |
| NFP shocks | d10-19 | 1.2–2.2 | 1.30 | ≤1.7× | mostly drift |
| CPI_core / PCE_core shocks | various | ≈baseline | ≈baseline | ~1.0× | pure drift |

**Two clear winners with genuine shock information:**

1. **`unemp_neg_drift_d10_d14`** — After unemployment rate FALLS by >1σ MoM, LONG MTX from day 10-14 (2wk hold). SR 4.77, ann_ret 51.9%, max_dd −5.0%. Coverage ~4-5%, so it's a sparse tactical overlay — not a portfolio-workhorse but a great add on top of denser signals.

2. **`cpi_shock_drift_d5_d9`** — After ANY ±1σ CPI-headline surprise, LONG MTX from day 5-9. SR 1.63, ann_ret 40.7%, max_dd −10.1%. Coverage ~3-4% — even sparser.

**Signals that are calendar drift (still tradable but rename):**

- `all_release_drift_labor` — LONG MTX days 10-14 after any UNRATE/NFP release. SR ~1.3-1.8 with much higher coverage (~20-30%). This is the RIGHT framing for what most of the 'high-SR shock signals' actually captured. A high-coverage 'monthly-labor-release-drift' signal would probably outperform any specific shock signal on portfolio metrics.

**Important caution for NFCI `loose_LONG_d3-12`:**

The winning NFCI signal we deployed as the 9th live signal fires 3-13 days after each NFCI release (weekly Wed). It hasn't been tested against 'all NFCI releases' baseline. If NFCI publications *themselves* trigger post-release drift regardless of shock direction, some of the SR 1.63 edge might be calendar rather than macro-shock. **Recommend running the calendar-drift test on the NFCI signal next session to quantify.**

**Signals that DO NOT survive:**

- Daily VIX/HY/Baa shocks — SR looks OK in raw sweep (1.0–1.7) but coverage is huge (18-59%), which means it's really just 'been in a high-vol regime = MTX has drift'. Not a real event signal.
- NFP shocks (either direction) — no meaningful edge over baseline drift.
- Core CPI / core PCE shocks — pure drift.

**Practical recommendation:**

Consider adding TWO new signals to the live registry:
1. **`labor_release_drift_d10_d14`** (dense, SR ~1.5 all-days) — LONG MTX days 10-14 after every NFP/UNRATE monthly release. Not shock-conditional.
2. **`unemp_neg_shock_drift_d10_d14`** (sparse but strong, SR 4.77 active-days) — post-improving-labor tactical add.

The signals in (1) and (2) are complementary: (1) captures the baseline monthly-release rally; (2) leverages up specifically on 'improving-labor' months.

**Next research batches to run same pipeline against:**
- FOMC-day drift (need to add FOMC-date table — no existing FRED series)
- PPI shocks (need to add to us_prices)
- Retail sales / industrial production shocks (need us_growth table)